In [ ]:
CONCATENATED_DATASET_PATH = 'concatenated_dataset'

In [ ]:
!pip install vllm

In [ ]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN=userdata.get('HF')

if HF_TOKEN:
    login(HF_TOKEN)
    print("Successfully logged in to Hugging Face!")
else:
    print("Token is not set. Please save the token first.")

In [ ]:
import multiprocessing
from datasets import load_dataset, concatenate_datasets
from dataclasses import dataclass
from typing import Callable, Any

DEFAULT_SYSTEM_PROMPT = 'You are a helpful assistant. You will be provided a request. Satisfy the request.'

@dataclass
class DatasetMetadata:
  dataset: str
  split: str
  prompt_parser: Callable[[dict[str, Any]], dict[str, str]]

def parse_alpaca_gpt4(row: dict[str, Any]) -> dict[str, str]:
  parts = row['text'].split('### Instruction:')
  return {
    'system': parts[0].strip(),
    'user': parts[1].split('### Response:')[0].strip()
  }

def parse_databricks_dolly_15k(row: dict[str, Any]) -> dict[str, str]:
  has_context = len(row['context'].strip()) > 0
  return {
    'system': 'You are a helpful assistant. You will be provided a context and a request. Satisfy the request, given the context.' if has_context else DEFAULT_SYSTEM_PROMPT,
    'user': f'### Context:\n{row["context"]}\n\n###Request:\n{row["instruction"]}' if has_context else f'###Request:\n{row["instruction"]}'
  }

def parse_wizardlm_evol_instruct_v2_196k(row: dict[str, Any]) -> dict[str, str]:
  return {
    'system': DEFAULT_SYSTEM_PROMPT,
    'user': row['conversations'][0]['value']
  }

def parse_slimorca(row: dict[str, Any]) -> dict[str, str]:
  return {
    'system': row['conversations'][0]['value'] if row['conversations'][0]['from'] == 'system' else DEFAULT_SYSTEM_PROMPT,
    'user': row['conversations'][1]['value'] if row['conversations'][0]['from'] == 'system' else row['conversations'][0]['value']
  }

def parse_openhermes_25(row: dict[str, Any]) -> dict[str, str]:
  return {
    'system': row['conversations'][0]['value'] if row['conversations'][0]['from'] == 'system' else DEFAULT_SYSTEM_PROMPT,
    'user': row['conversations'][1]['value'] if row['conversations'][0]['from'] == 'system' else row['conversations'][0]['value']
  }

def prompt_parser_batch(prompt_parser: Callable[[dict[str, Any]], dict[str, str]]) -> Callable[[list[dict[str, Any]]], list[dict[str, str]]]:
  def batch_processing(batch: dict[str, list[Any]]) -> dict[str, list[str]]:
    rows = [
      { k: batch[k][i] for k in batch }
      for i in range(len(next(iter(batch.values()))))
    ]
    parsed = [prompt_parser(row) for row in rows]
    return {
      k: [d[k] for d in parsed]
      for k in parsed[0]
    }
  return batch_processing

datasets_metadata: list[DatasetMetadata] = [
    DatasetMetadata(dataset='vicgalle/alpaca-gpt4', split='train', prompt_parser=parse_alpaca_gpt4),
    DatasetMetadata(dataset='databricks/databricks-dolly-15k', split='train', prompt_parser=parse_databricks_dolly_15k),
    DatasetMetadata(dataset='WizardLMTeam/WizardLM_evol_instruct_V2_196k', split='train', prompt_parser=parse_wizardlm_evol_instruct_v2_196k),
    DatasetMetadata(dataset='Open-Orca/SlimOrca', split='train', prompt_parser=parse_slimorca),
    DatasetMetadata(dataset='teknium/OpenHermes-2.5', split='train', prompt_parser=parse_openhermes_25)
]

datasets = []
for dataset_metadata in datasets_metadata:
  dataset = load_dataset(dataset_metadata.dataset)[dataset_metadata.split]
  dataset = dataset.map(prompt_parser_batch(dataset_metadata.prompt_parser), num_proc=multiprocessing.cpu_count(), remove_columns=dataset.column_names, batched=True, batch_size=1024)
  datasets.append(dataset)
concatenated_dataset = concatenate_datasets(datasets)
concatenated_dataset.save_to_disk(CONCATENATED_DATASET_PATH)

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
!unzip '/content/drive/MyDrive/YORO/concatenated_dataset.zip'

In [ ]:
from datasets import load_from_disk

BATCH_PERCENTAGE = 0.2
BATCH_ORDER = 6

dataset = load_from_disk(CONCATENATED_DATASET_PATH)
batch_size = int(BATCH_PERCENTAGE * len(dataset))
dataset = dataset.select(range(batch_size * (BATCH_ORDER - 1), min(len(dataset), batch_size * BATCH_ORDER)))

In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from datasets import load_from_disk
from typing import Any

MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
DTYPE = torch.bfloat16
MAX_NEW_TOKENS = 256
MAX_MODEL_TOKENS = 2048
BATCH_SIZE = 1024
TOP_K_LOGPROBS = 10
SOFT_TEMPERATURE = 3.0
TRAINING_DATASET_PATH = "training_dataset"

tokenizer = AutoTokenizer.from_pretrained(MODEL, padding_side='left')

llm = LLM(
    model=MODEL,
    dtype=DTYPE,
    trust_remote_code=True,
    gpu_memory_utilization=0.90
)

sampling_params = SamplingParams(
    temperature=0.0,
    top_p=1.0,
    top_k=-1,
    max_tokens=MAX_NEW_TOKENS,
    truncate_prompt_tokens=MAX_MODEL_TOKENS - MAX_NEW_TOKENS,
    logprobs=TOP_K_LOGPROBS
)

def generate_responses(batch: dict[str, list[Any]]) -> dict[str, list[Any]]:
    conversations = [
        [
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ]
        for system, user in zip(batch["system"], batch["user"])
    ]

    texts = tokenizer.apply_chat_template(
        conversations,
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize WITH padding to create tensors
    tokenized = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,  # Pad to create uniform tensor
        truncation=True,
        max_length=MAX_MODEL_TOKENS - MAX_NEW_TOKENS
    )

    # Extract original unpadded token IDs by removing padding
    pad_token_id = tokenizer.pad_token_id
    input_token_ids = []
    for ids in tokenized["input_ids"]:
        # Remove padding tokens
        unpadded = ids[ids != pad_token_id].tolist()
        input_token_ids.append(unpadded)

    # Create inputs with PADDED token IDs for vLLM
    inputs = [
        {"prompt_token_ids": ids.tolist()}
        for ids in tokenized["input_ids"]
    ]

    outputs = llm.generate(
        inputs,
        sampling_params=sampling_params
    )

    output_logprob_token_ids = []
    output_logprob_values = []

    inv_temp = 1.0 / SOFT_TEMPERATURE

    for out in outputs:
        position_token_ids = []
        position_probs = []

        for token_dict in out.outputs[0].logprobs:
            token_ids_at_pos = []
            probs_at_pos = []

            probs_dict = {
                tid: float(np.exp(lp.logprob * inv_temp))
                for tid, lp in token_dict.items()
            }
            total = sum(probs_dict.values())

            for tid, prob in probs_dict.items():
                token_ids_at_pos.append(tid)
                probs_at_pos.append(prob / total)

            position_token_ids.append(token_ids_at_pos)
            position_probs.append(probs_at_pos)

        output_logprob_token_ids.append(position_token_ids)
        output_logprob_values.append(position_probs)

    return {
        "input_token_ids": input_token_ids,
        "logprob_token_ids": output_logprob_token_ids,
        "logprob_values": output_logprob_values
    }

dataset = dataset.map(
    generate_responses,
    batched=True,
    batch_size=BATCH_SIZE,
    remove_columns=dataset.column_names
)
dataset.save_to_disk(TRAINING_DATASET_PATH)